# Uber Fare Prediction: Linear Regression and Random Forest

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
data=pd.read_csv("/home/student/Downloads/uber.csv")

In [ ]:
data

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.columns

In [ ]:
data=data.drop(['Unnamed: 0'], axis=1)

In [ ]:
data=data.drop(['key'], axis=1)

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.dtypes

In [ ]:
data.isnull().sum()

In [ ]:
data['dropoff_longitude'].fillna(value=data['dropoff_longitude'].mean(), inplace=True)
data['dropoff_latitude'].fillna(value=data['dropoff_latitude'].median(), inplace=True)

In [ ]:
data.isnull().sum()

In [ ]:
data.dtypes

In [ ]:
data.pickup_datetime=pd.to_datetime(data.pickup_datetime, errors='coerce')

In [ ]:
data.dtypes

In [ ]:
data= data.assign(hour = data.pickup_datetime.dt.hour,
               day= data.pickup_datetime.dt.day,
               month = data.pickup_datetime.dt.month,
               year = data.pickup_datetime.dt.year,
               dayofweek = data.pickup_datetime.dt.dayofweek)

In [ ]:
data.head()

In [ ]:
data=data.drop(['pickup_datetime'], axis=1)

In [ ]:
data.head()

In [ ]:
data.plot(kind = "box", subplots = True, layout = (7,2), figsize=(15,20))

In [ ]:
#Using the InterQuartile Range to fill the values
def remove_outlier(data1, col):
    Q1 = data1[col].quantile(0.25)
    Q3 = data1[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_whisker = Q1 - 1.5*IQR
    upper_whisker = Q3 + 1.5*IQR
    data[col] = np.clip(data1[col], lower_whisker, upper_whisker)
    return data1

def treat_outliers_all(data1, col_list):
    for c in col_list:
        data1 = remove_outlier(data, c)
    return data1

In [ ]:
data=treat_outliers_all(data, data.iloc[: , 0::])

In [ ]:
data.plot(kind = "box", subplots = True, layout = (7,2), figsize=(15,20))

In [ ]:
#3 check correlation
corr = data.corr()
corr

In [ ]:
sns.heatmap(data.corr(), annot=True) #correlation Heatmap (light values means highly correlated)

In [ ]:
from math import *
# function to calculate the travel distance from the longitudes and latitudes
def distance_transform(longitude1, latitude1, longitude2, latitude2):
    travel_dist= []

    for pos in range(len(longitude1)):
        long1,lati1,long2,lati2= map(radians,[longitude1[pos],latitude1[pos],longitude2[pos],latitude2[pos]])
        dist_long= long2- long1
        dist_lati= lati2- lati1
        a= sin(dist_lati/2)**2 + cos(lati1) * cos(lati2) * sin(dist_long/2)**2
        c= 2 * asin(sqrt(a))*6371
        travel_dist.append(c)

    return travel_dist

In [ ]:
data['dist_travel_km'] = distance_transform(data['pickup_longitude'].to_numpy(),
                                            data['pickup_latitude'].to_numpy(),
                                            data['dropoff_longitude'].to_numpy(),
                                            data['dropoff_latitude'].to_numpy())
data.head()

## Linear regression

In [ ]:
x = data[['pickup_longitude','pickup_latitude','dropoff_longitude','dropoff_latitude','passenger_count','hour','day','month','year','dayofweek','dist_travel_km']]
y = data['fare_amount']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size = 0.33)

In [ ]:
from sklearn.linear_model import LinearRegression
regression = LinearRegression()

In [ ]:
regression.fit(X_train, y_train)

In [ ]:
regression.intercept_ #To find the linear intercept

In [ ]:
regression.coef_ #To find the linear coefficient

In [ ]:
pred = regression.predict(X_test)
pred

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r2_score(y_test,pred)

In [ ]:
MSE = mean_squared_error(y_test, pred)
MSE

In [ ]:
RMSE = np.sqrt(MSE)
RMSE

## Random forest

In [ ]:
#RANDOM FOREST
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf = RandomForestRegressor(n_estimators=15)
rf.fit(X_train, y_train)

In [ ]:
pred = rf.predict(X_test)
pred

In [ ]:
R = r2_score(y_test, pred)
R

In [ ]:
MSE = mean_squared_error(y_test, pred)
MSE

In [ ]:
RMSE = np.sqrt(MSE)
RMSE